# Model 4: Best Offer Prediction (Retention Optimization by Segment)

#Importing the required packages

In [0]:
%restart_python 

In [0]:
!pip install statsmodels
!pip install xgboost
!pip install lightgbm

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import joblib

# Importing the data

In [0]:
VOLUME_PATH = "/Volumes/workspace/default/raw_data/"

In [0]:
demographics = pd.read_csv(VOLUME_PATH + "customer_demographics.csv")
location = pd.read_csv(VOLUME_PATH + "customer_location.csv")
services = pd.read_csv(VOLUME_PATH + "customer_services.csv")
account_status = pd.read_csv(VOLUME_PATH + "customer_account_status.csv")
zipcode_population = pd.read_csv(VOLUME_PATH + "zipcode_population.csv")

cluster_data = pd.read_csv(VOLUME_PATH + "customer_cluster_data.csv")

# Cleaning
(Same cleaning steps as Model 1 / Model 2, so the merged table lines up column-for-column.)

In [0]:
services['Internet_Type_Clean'] = services['Internet Type'].fillna('No Internet Service')
services['Offer_Clean'] = services['Offer'].fillna('No Offer')

In [0]:
def missing_value_imp(x):
  if x.dtype == 'int' or x.dtype == 'float':
    x = x.fillna(x.mean())
  else:
    x = x.fillna(x.mode()[0])
  return x

In [0]:
services = services.apply(missing_value_imp)

In [0]:
account_status['Churn_Category_Clean'] = account_status['Churn Category'].fillna('Not Churned')
account_status['Churn_Reason_Clean'] = account_status['Churn Reason'].fillna('Not Churned')

In [0]:
account_status['Has_Discount'] = np.where(account_status['Monthly Charge'] < 0, 1, 0)
account_status['Monthly_Discount_Amount'] = np.where(
    account_status['Monthly Charge'] < 0, account_status['Monthly Charge'].abs(), 0)

#Merge into one master table

In [0]:
data = demographics.merge(location, on='Customer ID', how='left')
data = data.merge(services, on='Customer ID', how='left')
data = data.merge(account_status, on='Customer ID', how='left')
data = data.merge(zipcode_population, on='Zip Code', how='left')

In [0]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 45 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Customer ID                        7043 non-null   object 
 1   Gender                             7043 non-null   object 
 2   Age                                7043 non-null   int64  
 3   Married                            7043 non-null   object 
 4   Number of Dependents               7043 non-null   int64  
 5   City                               7043 non-null   object 
 6   Zip Code                           7043 non-null   int64  
 7   Latitude                           7043 non-null   float64
 8   Longitude                          7043 non-null   float64
 9   Offer                              7043 non-null   object 
 10  Phone Service                      7043 non-null   object 
 11  Avg Monthly Long Distance Charges  7043 non-null   float

#Model 4: Best Offer Prediction

In [0]:
data.drop(columns=['Offer','Internet Type','Churn Category','Churn Reason'], inplace=True)

In [0]:
data.columns = data.columns.str.replace(' ','_')

In [0]:
data.head(3)

,Customer_ID,Gender,Age,Married,Number_of_Dependents,City,Zip_Code,Latitude,Longitude,Phone_Service,Avg_Monthly_Long_Distance_Charges,Multiple_Lines,Internet_Service,Avg_Monthly_GB_Download,Online_Security,Online_Backup,Device_Protection_Plan,Premium_Tech_Support,Streaming_TV,Streaming_Movies,Streaming_Music,Unlimited_Data,Internet_Type_Clean,Offer_Clean,Number_of_Referrals,Tenure_in_Months,Contract,Paperless_Billing,Payment_Method,Monthly_Charge,Total_Charges,Total_Refunds,Total_Extra_Data_Charges,Total_Long_Distance_Charges,Total_Revenue,Customer_Status,Churn_Category_Clean,Churn_Reason_Clean,Has_Discount,Monthly_Discount_Amount,Population
0,0002-ORFBO,Female,37,Yes,0,Frazier Park,93225,34.827662,-118.999073,Yes,42.39,No,Yes,16.0,No,Yes,No,Yes,Yes,No,No,Yes,Cable,No Offer,2,9,One Year,Yes,Credit Card,65.6,593.30,0.00,0,381.51,974.81,Stayed,Not Churned,Not Churned,0,0.0,4498
1,0003-MKNFE,Male,46,No,0,Glendale,91206,34.162515,-118.203869,Yes,10.69,Yes,Yes,10.0,No,No,No,No,No,Yes,Yes,No,Cable,No Offer,0,9,Month-to-Month,No,Credit Card,-4.0,542.40,38.33,10,96.21,610.28,Stayed,Not Churned,Not Churned,1,4.0,31297
2,0004-TLHLJ,Male,50,No,0,Costa Mesa,92627,33.645672,-117.922613,Yes,33.65,No,Yes,30.0,No,No,Yes,No,No,No,No,Yes,Fiber Optic,Offer E,0,4,Month-to-Month,Yes,Bank Withdrawal,73.9,280.85,0.00,0,134.60,415.45,Churned,Competitor,Competitor had better devices,0,0.0,62069


# Encoding categorical columns

In [0]:
data['Gender'] = pd.get_dummies(data['Gender'], drop_first=True, dtype='int')
data['Married'] = np.where(data['Married'] == 'Yes', 1, 0)
data['Phone_Service'] = np.where(data['Phone_Service'] == 'Yes', 1, 0)
data['Multiple_Lines'] = np.where(data['Multiple_Lines'] == 'Yes', 1, 0)
data['Internet_Service'] = np.where(data['Internet_Service'] == 'Yes', 1, 0)
data['Online_Security'] = np.where(data['Online_Security'] == 'Yes', 1, 0)
data['Online_Backup'] = np.where(data['Online_Backup'] == 'Yes', 1, 0)
data['Device_Protection_Plan'] = np.where(data['Device_Protection_Plan'] == 'Yes', 1, 0)
data['Premium_Tech_Support'] = np.where(data['Premium_Tech_Support'] == 'Yes', 1, 0)
data['Streaming_TV'] = np.where(data['Streaming_TV'] == 'Yes', 1, 0)
data['Streaming_Movies'] = np.where(data['Streaming_Movies'] == 'Yes', 1, 0)
data['Streaming_Music'] = np.where(data['Streaming_Music'] == 'Yes', 1, 0)
data['Unlimited_Data'] = np.where(data['Unlimited_Data'] == 'Yes', 1, 0)
data = pd.concat([data, pd.get_dummies(data['Internet_Type_Clean'], drop_first=True, dtype='int', prefix='Internet_Type_Clean')], axis=1)
data.drop('Internet_Type_Clean', axis=1, inplace=True)
data = pd.concat([data, pd.get_dummies(data['Offer_Clean'], drop_first=True, dtype='int', prefix='Offer_Clean')], axis=1)
data.drop('Offer_Clean', axis=1, inplace=True)
data = pd.concat([data, pd.get_dummies(data['Contract'], drop_first=True, dtype='int', prefix='Contract')], axis=1)
data.drop('Contract', axis=1, inplace=True)
data['Paperless_Billing'] = np.where(data['Paperless_Billing'] == 'Yes', 1, 0)
data = pd.concat([data, pd.get_dummies(data['Payment_Method'], drop_first=True, dtype='int', prefix='Payment_Method')], axis=1)
data.drop('Payment_Method', axis=1, inplace=True)
data['Customer_Status'] = np.where(data['Customer_Status'] == 'Churned', 1, 0)

In [0]:
data.rename(columns={'Customer_Status': 'Churned'}, inplace=True)
data.columns = data.columns.str.replace(' ','_')

# Bring in customer segments from Model 3

In [0]:
cluster_data.columns = cluster_data.columns.str.replace(' ','_')
cluster_data.head()

,Customer_ID,Cluster
0,0002-ORFBO,2
1,0003-MKNFE,4
2,0004-TLHLJ,2
3,0011-IGKFF,1
4,0013-EXCHZ,2


In [0]:
data = data.merge(cluster_data[['Customer_ID','Cluster']], on='Customer_ID', how='left')

In [0]:
data['Cluster'].isnull().sum()

np.int64(0)

In [0]:
cluster_dummies = pd.get_dummies(data['Cluster'], prefix='Cluster', drop_first=True, dtype='int')
data = pd.concat([data, cluster_dummies], axis=1)

In [0]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 54 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   Customer_ID                              7043 non-null   object 
 1   Gender                                   7043 non-null   int64  
 2   Age                                      7043 non-null   int64  
 3   Married                                  7043 non-null   int64  
 4   Number_of_Dependents                     7043 non-null   int64  
 5   City                                     7043 non-null   object 
 6   Zip_Code                                 7043 non-null   int64  
 7   Latitude                                 7043 non-null   float64
 8   Longitude                                7043 non-null   float64
 9   Phone_Service                            7043 non-null   int64  
 10  Avg_Monthly_Long_Distance_Charges        7043 no

In [0]:
X = data.drop(['Customer_ID','Churned','City','Zip_Code','Cluster','Churn_Category_Clean','Churn_Reason_Clean'] , axis=1)
y = data['Churned']

In [0]:
X.columns

Index(['Gender', 'Age', 'Married', 'Number_of_Dependents', 'Latitude',
       'Longitude', 'Phone_Service', 'Avg_Monthly_Long_Distance_Charges',
       'Multiple_Lines', 'Internet_Service', 'Avg_Monthly_GB_Download',
       'Online_Security', 'Online_Backup', 'Device_Protection_Plan',
       'Premium_Tech_Support', 'Streaming_TV', 'Streaming_Movies',
       'Streaming_Music', 'Unlimited_Data', 'Number_of_Referrals',
       'Tenure_in_Months', 'Paperless_Billing', 'Monthly_Charge',
       'Total_Charges', 'Total_Refunds', 'Total_Extra_Data_Charges',
       'Total_Long_Distance_Charges', 'Total_Revenue', 'Has_Discount',
       'Monthly_Discount_Amount', 'Population', 'Internet_Type_Clean_DSL',
       'Internet_Type_Clean_Fiber_Optic',
       'Internet_Type_Clean_No_Internet_Service', 'Offer_Clean_Offer_A',
       'Offer_Clean_Offer_B', 'Offer_Clean_Offer_C', 'Offer_Clean_Offer_D',
       'Offer_Clean_Offer_E', 'Contract_One_Year', 'Contract_Two_Year',
       'Payment_Method_Credit_Card',

# Splitting the data

In [0]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

# Feature engineering (VIF)

In [0]:
vif = pd.DataFrame()
vif['feature'] = x_train.columns
vif['score'] = [variance_inflation_factor(x_train.values, i) for i in range(len(x_train.columns))]

/local_disk0/.ephemeral_nfs/envs/pythonEnv-e643bad3-0a40-407d-84ff-0b5afa4ddc9a/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/local_disk0/.ephemeral_nfs/envs/pythonEnv-e643bad3-0a40-407d-84ff-0b5afa4ddc9a/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/local_disk0/.ephemeral_nfs/envs/pythonEnv-e643bad3-0a40-407d-84ff-0b5afa4ddc9a/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:235: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  r_sq = OLS(x_i, x_noti).fit().rsquared
/local_disk0/.ephemeral_nfs/envs/pythonEnv-e643bad3-0a40-407d-84ff-0b5afa4ddc9a/lib/p

In [0]:
vif.loc[vif['score']<5, 'feature'].values

array(['Gender', 'Age', 'Married', 'Number_of_Dependents',
       'Phone_Service', 'Avg_Monthly_Long_Distance_Charges',
       'Multiple_Lines', 'Avg_Monthly_GB_Download', 'Online_Security',
       'Online_Backup', 'Device_Protection_Plan', 'Premium_Tech_Support',
       'Streaming_TV', 'Streaming_Music', 'Unlimited_Data',
       'Number_of_Referrals', 'Paperless_Billing', 'Population',
       'Internet_Type_Clean_DSL', 'Offer_Clean_Offer_A',
       'Offer_Clean_Offer_B', 'Offer_Clean_Offer_C',
       'Offer_Clean_Offer_D', 'Offer_Clean_Offer_E', 'Contract_One_Year',
       'Contract_Two_Year', 'Payment_Method_Credit_Card',
       'Payment_Method_Mailed_Check'], dtype=object)

In [0]:
data.columns

Index(['Customer_ID', 'Gender', 'Age', 'Married', 'Number_of_Dependents',
       'City', 'Zip_Code', 'Latitude', 'Longitude', 'Phone_Service',
       'Avg_Monthly_Long_Distance_Charges', 'Multiple_Lines',
       'Internet_Service', 'Avg_Monthly_GB_Download', 'Online_Security',
       'Online_Backup', 'Device_Protection_Plan', 'Premium_Tech_Support',
       'Streaming_TV', 'Streaming_Movies', 'Streaming_Music', 'Unlimited_Data',
       'Number_of_Referrals', 'Tenure_in_Months', 'Paperless_Billing',
       'Monthly_Charge', 'Total_Charges', 'Total_Refunds',
       'Total_Extra_Data_Charges', 'Total_Long_Distance_Charges',
       'Total_Revenue', 'Churned', 'Churn_Category_Clean',
       'Churn_Reason_Clean', 'Has_Discount', 'Monthly_Discount_Amount',
       'Population', 'Internet_Type_Clean_DSL',
       'Internet_Type_Clean_Fiber_Optic',
       'Internet_Type_Clean_No_Internet_Service', 'Offer_Clean_Offer_A',
       'Offer_Clean_Offer_B', 'Offer_Clean_Offer_C', 'Offer_Clean_Offer_D',
  

In [0]:
base_features = ['Gender', 'Age', 'Married', 'Number_of_Dependents',
       'Phone_Service', 'Avg_Monthly_Long_Distance_Charges',
       'Multiple_Lines', 'Avg_Monthly_GB_Download', 'Online_Security',
       'Online_Backup', 'Device_Protection_Plan', 'Premium_Tech_Support',
       'Streaming_TV', 'Streaming_Music', 'Unlimited_Data',
       'Number_of_Referrals', 'Paperless_Billing', 'Population',
       'Internet_Type_Clean_DSL', 'Offer_Clean_Offer_A',
       'Offer_Clean_Offer_B', 'Offer_Clean_Offer_C',
       'Offer_Clean_Offer_D', 'Offer_Clean_Offer_E', 'Contract_One_Year',
       'Contract_Two_Year', 'Payment_Method_Credit_Card',
       'Payment_Method_Mailed_Check']

offer_cols = ['Offer_Clean_Offer_A', 'Offer_Clean_Offer_B', 'Offer_Clean_Offer_C',
              'Offer_Clean_Offer_D', 'Offer_Clean_Offer_E']
cluster_cols = ['Cluster_1', 'Cluster_2', 'Cluster_3', 'Cluster_4']

x = data[base_features + cluster_cols]

# Splitting the data (final feature set)

In [0]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=123)

# Checking class imbalance

In [0]:
neg, pos = np.bincount(y_train)

In [0]:
imbalance_ratio = neg / pos
imbalance_ratio

np.float64(2.7761394101876675)

# Model building

# Logistic regression

In [0]:
par_grid_lr = {'C': [0.01, 0.1, 1]}

In [0]:
grid_lr = GridSearchCV(LogisticRegression(max_iter=500, class_weight='balanced'),
                        param_grid=par_grid_lr, cv=3, scoring='roc_auc', n_jobs=-1)
grid_lr = grid_lr.fit(x_train, y_train)

/databricks/python/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/databricks/python/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression

In [0]:
lr = grid_lr.best_estimator_
lr.fit(x_train, y_train)

LogisticRegression(C=1, class_weight='balanced', max_iter=500)

In [0]:
print(classification_report(y_train, lr.predict(x_train)))
print(classification_report(y_test, lr.predict(x_test)))

              precision    recall  f1-score   support

           0       0.94      0.77      0.84      4142
           1       0.57      0.86      0.68      1492

    accuracy                           0.79      5634
   macro avg       0.75      0.81      0.76      5634
weighted avg       0.84      0.79      0.80      5634

              precision    recall  f1-score   support

           0       0.93      0.75      0.83      1032
           1       0.55      0.84      0.67       377

    accuracy                           0.78      1409
   macro avg       0.74      0.80      0.75      1409
weighted avg       0.83      0.78      0.79      1409



# Random forest

In [0]:
par_grid_rf = {'n_estimators': [100], 'max_depth': [6, 10]}

In [0]:
grid_rf = GridSearchCV(RandomForestClassifier(n_jobs=-1, random_state=123, class_weight='balanced'),
                        param_grid=par_grid_rf, cv=3, scoring='roc_auc', n_jobs=-1)
grid_rf = grid_rf.fit(x_train, y_train)

In [0]:
rf = grid_rf.best_estimator_
rf.fit(x_train, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=10, n_jobs=-1,
                       random_state=123)

In [0]:
print(classification_report(y_train, rf.predict(x_train)))
print(classification_report(y_test, rf.predict(x_test)))

              precision    recall  f1-score   support

           0       0.98      0.84      0.91      4142
           1       0.68      0.95      0.80      1492

    accuracy                           0.87      5634
   macro avg       0.83      0.90      0.85      5634
weighted avg       0.90      0.87      0.88      5634

              precision    recall  f1-score   support

           0       0.92      0.81      0.86      1032
           1       0.60      0.81      0.69       377

    accuracy                           0.81      1409
   macro avg       0.76      0.81      0.78      1409
weighted avg       0.84      0.81      0.81      1409



# XGBoost

In [0]:
par_grid_xgb = {'n_estimators': [100], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}

In [0]:
grid_xgb = GridSearchCV(XGBClassifier(eval_metric='logloss', random_state=123, scale_pos_weight=imbalance_ratio),
                         param_grid=par_grid_xgb, cv=3, scoring='roc_auc', n_jobs=-1)
grid_xgb = grid_xgb.fit(x_train, y_train)

In [0]:
xgb = grid_xgb.best_estimator_
xgb.fit(x_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [0]:
print(classification_report(y_train, xgb.predict(x_train)))
print(classification_report(y_test, xgb.predict(x_test)))

              precision    recall  f1-score   support

           0       0.95      0.79      0.86      4142
           1       0.61      0.89      0.72      1492

    accuracy                           0.82      5634
   macro avg       0.78      0.84      0.79      5634
weighted avg       0.86      0.82      0.83      5634

              precision    recall  f1-score   support

           0       0.94      0.78      0.85      1032
           1       0.59      0.86      0.70       377

    accuracy                           0.80      1409
   macro avg       0.76      0.82      0.77      1409
weighted avg       0.84      0.80      0.81      1409



# LightGBM

In [0]:
par_grid_lgb = {'n_estimators': [100], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]}

In [0]:
grid_lgb = GridSearchCV(LGBMClassifier(random_state=123, verbose=-1, class_weight='balanced'),
                         param_grid=par_grid_lgb, cv=3, scoring='roc_auc', n_jobs=-1)
grid_lgb = grid_lgb.fit(x_train, y_train)

In [0]:
lgbm = grid_lgb.best_estimator_
lgbm.fit(x_train, y_train)

LGBMClassifier(class_weight='balanced', max_depth=3, random_state=123,
               verbose=-1)

In [0]:
print(classification_report(y_train, lgbm.predict(x_train)))
print(classification_report(y_test, lgbm.predict(x_test)))

              precision    recall  f1-score   support

           0       0.95      0.79      0.86      4142
           1       0.61      0.89      0.72      1492

    accuracy                           0.82      5634
   macro avg       0.78      0.84      0.79      5634
weighted avg       0.86      0.82      0.83      5634

              precision    recall  f1-score   support

           0       0.94      0.77      0.85      1032
           1       0.58      0.85      0.69       377

    accuracy                           0.79      1409
   macro avg       0.76      0.81      0.77      1409
weighted avg       0.84      0.79      0.80      1409



# Comparing all the model predictions

In [0]:
score = pd.DataFrame([grid_lr.best_score_, grid_rf.best_score_, grid_xgb.best_score_, grid_lgb.best_score_])
name = pd.DataFrame(['logistic_regression', 'random_forest', 'xgboost', 'lightgbm'])
best_score = pd.concat([name, score], axis=1)
best_score.columns = ['model', 'roc_auc']
best_score.sort_values('roc_auc', ascending=False)

,model,roc_auc
2,xgboost,0.898071
3,lightgbm,0.897471
1,random_forest,0.889692
0,logistic_regression,0.881696


In [0]:
best_model = grid_xgb.best_estimator_

# Predicting the best offer per customer

In [0]:
offer_cols = ['Offer_Clean_Offer_A', 'Offer_Clean_Offer_B', 'Offer_Clean_Offer_C',
              'Offer_Clean_Offer_D', 'Offer_Clean_Offer_E']

In [0]:
sim_base = x.copy()
sim_base

,Gender,Age,Married,Number_of_Dependents,Phone_Service,Avg_Monthly_Long_Distance_Charges,Multiple_Lines,Avg_Monthly_GB_Download,Online_Security,Online_Backup,Device_Protection_Plan,Premium_Tech_Support,Streaming_TV,Streaming_Music,Unlimited_Data,Number_of_Referrals,Paperless_Billing,Population,Internet_Type_Clean_DSL,Offer_Clean_Offer_A,Offer_Clean_Offer_B,Offer_Clean_Offer_C,Offer_Clean_Offer_D,Offer_Clean_Offer_E,Contract_One_Year,Contract_Two_Year,Payment_Method_Credit_Card,Payment_Method_Mailed_Check,Cluster_1,Cluster_2,Cluster_3,Cluster_4
0,0,37,1,0,1,42.390000,0,16.0,0,1,0,1,1,0,1,2,1,4498,0,0,0,0,0,0,1,0,1,0,0,1,0,0
1,1,46,0,0,1,10.690000,1,10.0,0,0,0,0,0,1,0,0,0,31297,0,0,0,0,0,0,0,0,1,0,0,0,0,1
2,1,50,0,0,1,33.650000,0,30.0,0,0,1,0,0,0,1,0,1,62069,0,0,0,0,0,1,0,0,0,0,0,1,0,0
3,1,78,1,0,1,27.820000,0,4.0,0,1,1,0,1,0,1,1,1,46677,0,0,0,0,1,0,0,0,0,0,1,0,0,0
4,0,75,1,0,1,7.380000,0,11.0,0,0,0,1,1,0,1,3,1,42853,0,0,0,0,0,0,0,0,1,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,20,0,0,1,46.680000,0,59.0,1,0,0,1,0,1,1,0,0,44652,1,0,0,0,1,0,1,0,1,0,0,0,0,0
7039,1,40,1,0,1,16.200000,1,17.0,0,0,0,0,0,1,1,1,1,16525,0,0,0,0,1,0,0,0,0,0,1,0,0,0
7040,1,22,0,0,1,18.620000,0,51.0,0,1,0,0,0,0,1,0,1,383,1,0,0,0,0,1,0,0,1,0,0,0,0,0
7041,1,21,1,0,1,2.120000,0,58.0,1,0,1,1,0,1,1,5,0,12173,0,1,0,0,0,0,0,1,1,0,0,0,0,0


In [0]:
offer_cols

['Offer_Clean_Offer_A',
 'Offer_Clean_Offer_B',
 'Offer_Clean_Offer_C',
 'Offer_Clean_Offer_D',
 'Offer_Clean_Offer_E']

In [0]:
# --- Scenario: No Offer ---
sim_no_offer = sim_base.copy()
sim_no_offer[offer_cols] = 0                     
prob_no_offer = best_model.predict_proba(sim_no_offer)[:, 1]

# --- Scenario: Offer A ---
sim_offer_a = sim_base.copy()
sim_offer_a[offer_cols] = 0                      
sim_offer_a['Offer_Clean_Offer_A'] = 1            
prob_offer_a = best_model.predict_proba(sim_offer_a)[:, 1]

# --- Scenario: Offer B ---
sim_offer_b = sim_base.copy()
sim_offer_b[offer_cols] = 0
sim_offer_b['Offer_Clean_Offer_B'] = 1
prob_offer_b = best_model.predict_proba(sim_offer_b)[:, 1]

# --- Scenario: Offer C ---
sim_offer_c = sim_base.copy()
sim_offer_c[offer_cols] = 0
sim_offer_c['Offer_Clean_Offer_C'] = 1
prob_offer_c = best_model.predict_proba(sim_offer_c)[:, 1]

# --- Scenario: Offer D ---
sim_offer_d = sim_base.copy()
sim_offer_d[offer_cols] = 0
sim_offer_d['Offer_Clean_Offer_D'] = 1
prob_offer_d = best_model.predict_proba(sim_offer_d)[:, 1]

# --- Scenario: Offer E ---
sim_offer_e = sim_base.copy()
sim_offer_e[offer_cols] = 0
sim_offer_e['Offer_Clean_Offer_E'] = 1
prob_offer_e = best_model.predict_proba(sim_offer_e)[:, 1]


In [0]:
# Put all six columns of predictions side by side, one row per customer
offer_sim = pd.DataFrame({
    'No_Offer': prob_no_offer,
    'Offer_A':  prob_offer_a,
    'Offer_B':  prob_offer_b,
    'Offer_C':  prob_offer_c,
    'Offer_D':  prob_offer_d,
    'Offer_E':  prob_offer_e,
})

In [0]:
offer_sim

,No_Offer,Offer_A,Offer_B,Offer_C,Offer_D,Offer_E
0,0.218253,0.218253,0.152093,0.203852,0.167053,0.226172
1,0.673057,0.673057,0.540183,0.659333,0.679696,0.743562
2,0.758634,0.758634,0.627524,0.735296,0.713128,0.837145
3,0.980050,0.980050,0.971338,0.978286,0.976696,0.981423
4,0.876807,0.876807,0.830796,0.867153,0.872839,0.882970
...,...,...,...,...,...,...
7038,0.145060,0.145060,0.134011,0.130400,0.107501,0.212832
7039,0.918231,0.918231,0.783709,0.916358,0.920479,0.950567
7040,0.489677,0.489677,0.297656,0.458882,0.407402,0.583100
7041,0.031993,0.033387,0.023251,0.029420,0.025652,0.039731


# Best offer by customer segment

In [0]:
offer_sim['Customer_ID'] = data['Customer_ID'].values
offer_sim['Cluster'] = data['Cluster'].values

offer_cols_named = ['No_Offer', 'Offer_A', 'Offer_B', 'Offer_C', 'Offer_D', 'Offer_E']
offer_sim['Recommended_Offer'] = offer_sim[offer_cols_named].idxmin(axis=1)

offer_sim

,No_Offer,Offer_A,Offer_B,Offer_C,Offer_D,Offer_E,Customer_ID,Cluster,Recommended_Offer
0,0.218253,0.218253,0.152093,0.203852,0.167053,0.226172,0002-ORFBO,2,Offer_B
1,0.673057,0.673057,0.540183,0.659333,0.679696,0.743562,0003-MKNFE,4,Offer_B
2,0.758634,0.758634,0.627524,0.735296,0.713128,0.837145,0004-TLHLJ,2,Offer_B
3,0.980050,0.980050,0.971338,0.978286,0.976696,0.981423,0011-IGKFF,1,Offer_B
4,0.876807,0.876807,0.830796,0.867153,0.872839,0.882970,0013-EXCHZ,2,Offer_B
...,...,...,...,...,...,...,...,...,...
7038,0.145060,0.145060,0.134011,0.130400,0.107501,0.212832,9987-LUTYD,0,Offer_D
7039,0.918231,0.918231,0.783709,0.916358,0.920479,0.950567,9992-RRAMN,1,Offer_B
7040,0.489677,0.489677,0.297656,0.458882,0.407402,0.583100,9992-UJOEL,0,Offer_B
7041,0.031993,0.033387,0.023251,0.029420,0.025652,0.039731,9993-LHIEB,0,Offer_B


In [0]:
best_offer_by_segment = offer_sim.groupby('Cluster')[offer_cols_named].mean()
best_offer_by_segment['Best_Offer'] = best_offer_by_segment[offer_cols_named].idxmin(axis=1)
best_offer_by_segment['Customer_Count'] = offer_sim.groupby('Cluster').size()
best_offer_by_segment

,No_Offer,Offer_A,Offer_B,Offer_C,Offer_D,Offer_E,Best_Offer,Customer_Count
Cluster,,,,,,,,
0,0.447601,0.447807,0.352296,0.434669,0.427994,0.517064,Offer_B,897
1,0.421433,0.421591,0.328845,0.409432,0.401588,0.483505,Offer_B,2498
2,0.434855,0.435013,0.347817,0.423119,0.417940,0.493711,Offer_B,2591
3,0.090021,0.090090,0.059180,0.085602,0.077056,0.128690,Offer_B,973
4,0.355254,0.355409,0.275579,0.344334,0.333372,0.413346,Offer_B,84


In [0]:
cluster_profile = offer_sim.groupby('Cluster').mean(numeric_only=True)
cluster_profile['Customer_Count'] = offer_sim.groupby('Cluster').size()
cluster_profile

,No_Offer,Offer_A,Offer_B,Offer_C,Offer_D,Offer_E,Customer_Count
Cluster,,,,,,,
0,0.447601,0.447807,0.352296,0.434669,0.427994,0.517064,897
1,0.421433,0.421591,0.328845,0.409432,0.401588,0.483505,2498
2,0.434855,0.435013,0.347817,0.423119,0.417940,0.493711,2591
3,0.090021,0.090090,0.059180,0.085602,0.077056,0.128690,973
4,0.355254,0.355409,0.275579,0.344334,0.333372,0.413346,84


In [0]:
for name, val in list(globals().items()):
    if isinstance(val, pd.DataFrame):
        print(name, val.shape)

_ (5, 7)
__ (5, 7)
___ (5, 7)
demographics (7043, 5)
location (7043, 5)
services (7043, 18)
account_status (7043, 19)
zipcode_population (1671, 2)
cluster_data (7043, 2)
data (7043, 54)
_13 (3, 41)
_16 (5, 2)
cluster_dummies (7043, 4)
X (7043, 47)
x_train (5634, 32)
x_test (1409, 32)
vif (47, 2)
x (7043, 32)
score (4, 1)
name (4, 1)
best_score (4, 2)
_47 (4, 2)
sim_base (7043, 32)
_50 (7043, 32)
sim_no_offer (7043, 32)
sim_offer_a (7043, 32)
sim_offer_b (7043, 32)
sim_offer_c (7043, 32)
sim_offer_d (7043, 32)
sim_offer_e (7043, 32)
offer_sim (7043, 9)
_54 (7043, 9)
_55 (7043, 9)
best_offer_by_segment (5, 8)
_56 (5, 8)
cluster_profile (5, 7)
_59 (5, 7)
_60 (5, 7)
_61 (5, 7)
_62 (5, 7)


In [0]:
df_original = data.merge(cluster_data, on='Customer_ID', how='inner')
print(df_original.shape)   # sanity check — should still be (7043, 55)

(7043, 55)


In [0]:
df_original = data.merge(cluster_data, on='Customer_ID', how='inner')
df_original = df_original.drop(columns=['Cluster_x']).rename(columns={'Cluster_y': 'Cluster'})

cluster_profile = df_original.groupby('Cluster').mean(numeric_only=True)
cluster_profile['Customer_Count'] = df_original.groupby('Cluster').size()
cluster_profile.round(2)

,Gender,Age,Married,Number_of_Dependents,Zip_Code,Latitude,Longitude,Phone_Service,Avg_Monthly_Long_Distance_Charges,Multiple_Lines,Internet_Service,Avg_Monthly_GB_Download,Online_Security,Online_Backup,Device_Protection_Plan,Premium_Tech_Support,Streaming_TV,Streaming_Movies,Streaming_Music,Unlimited_Data,Number_of_Referrals,Tenure_in_Months,Paperless_Billing,Monthly_Charge,Total_Charges,Total_Refunds,Total_Extra_Data_Charges,Total_Long_Distance_Charges,Total_Revenue,Churned,Has_Discount,Monthly_Discount_Amount,Population,Internet_Type_Clean_DSL,Internet_Type_Clean_Fiber_Optic,Internet_Type_Clean_No_Internet_Service,Offer_Clean_Offer_A,Offer_Clean_Offer_B,Offer_Clean_Offer_C,Offer_Clean_Offer_D,Offer_Clean_Offer_E,Contract_One_Year,Contract_Two_Year,Payment_Method_Credit_Card,Payment_Method_Mailed_Check,Cluster_1,Cluster_2,Cluster_3,Cluster_4,Customer_Count
Cluster,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,0.49,24.29,0.44,0.18,93343.47,35.87,-119.49,0.88,26.07,0.46,1.00,58.94,0.40,0.45,0.44,0.40,0.49,0.52,0.64,0.85,1.65,32.36,0.66,76.09,2703.05,1.66,9.44,741.01,3451.84,0.30,0.00,0.01,24789.92,0.31,0.52,0.00,0.09,0.12,0.07,0.07,0.13,0.22,0.20,0.35,0.06,0.0,0.0,0.0,0.0,897
1,0.51,50.95,0.40,0.12,95143.73,38.48,-121.71,0.90,25.68,0.42,0.78,18.87,0.28,0.33,0.34,0.28,0.38,0.39,0.33,0.89,1.51,31.26,0.60,64.66,2211.65,1.88,6.31,733.67,2949.75,0.30,0.00,0.01,14519.84,0.23,0.45,0.22,0.06,0.11,0.05,0.09,0.11,0.21,0.25,0.37,0.06,1.0,0.0,0.0,0.0,2498
2,0.50,51.76,0.43,0.14,91872.70,34.00,-117.86,0.91,25.03,0.43,0.79,18.65,0.25,0.34,0.34,0.27,0.39,0.38,0.31,0.89,1.55,31.29,0.62,65.26,2253.29,2.08,7.01,722.06,2980.28,0.31,0.01,0.01,29040.93,0.22,0.46,0.21,0.07,0.11,0.06,0.08,0.13,0.23,0.24,0.37,0.05,0.0,1.0,0.0,0.0,2591
3,0.51,41.47,0.86,2.51,93678.73,36.52,-120.03,0.90,25.34,0.36,0.57,34.80,0.29,0.30,0.29,0.27,0.29,0.27,0.27,0.93,4.44,38.43,0.44,51.01,2168.47,2.07,5.36,877.88,3049.65,0.04,0.01,0.01,21023.41,0.22,0.24,0.43,0.09,0.14,0.06,0.09,0.07,0.23,0.42,0.51,0.05,0.0,0.0,1.0,0.0,973
4,0.57,48.04,0.45,0.54,93246.43,35.94,-119.64,0.90,23.88,0.40,0.77,27.04,0.29,0.33,0.27,0.35,0.33,0.33,0.35,0.87,2.00,29.77,0.52,-6.88,1942.78,2.95,8.45,636.71,2585.00,0.24,1.00,6.88,20491.38,0.33,0.32,0.23,0.06,0.06,0.05,0.07,0.13,0.24,0.27,0.46,0.06,0.0,0.0,0.0,1.0,84


In [0]:
cols_to_show = ['Age', 'Avg_Monthly_GB_Download', 'Total_Revenue', 'Number_of_Referrals', 'Tenure_in_Months', 'Churned', 'Customer_Count']
cluster_profile[cols_to_show]

,Age,Avg_Monthly_GB_Download,Total_Revenue,Number_of_Referrals,Tenure_in_Months,Churned,Customer_Count
Cluster,,,,,,,
0,24.286511,58.937781,3451.836522,1.646600,32.364548,0.302118,897
1,50.954363,18.873654,2949.750797,1.509207,31.259808,0.295436,2498
2,51.759552,18.646047,2980.280892,1.549981,31.294481,0.308761,2591
3,41.474820,34.798338,3049.649856,4.435766,38.434738,0.041110,973
4,48.035714,27.042967,2584.996905,2.000000,29.773810,0.238095,84


# Saving the model and results

In [0]:
joblib.dump(xgb, VOLUME_PATH + "best_offer_churn_model.pkl")
best_offer_by_segment.to_csv(VOLUME_PATH + "best_offer_by_segment.csv")
offer_sim[['Customer_ID', 'Cluster', 'Recommended_Offer']].to_csv(
    VOLUME_PATH + "customer_offer_recommendations.csv", index=False)

In [0]:
joblib.load(VOLUME_PATH + "best_offer_churn_model.pkl")

In [0]:
pd.read_csv(VOLUME_PATH + "best_offer_by_segment.csv")

In [0]:
pd.read_csv(VOLUME_PATH + "customer_offer_recommendations.csv")

#End